# RELAX0 P1 — 신호 규칙을 줄이면 얼마나 줄어드나 (무리 도착 세계, 규칙 A·B, 학습 없음)

학습 없는 계획기로, 같은 84상황에서 규칙만 바꿔 잽니다. 황색·전적색은 그대로입니다. 측정 방법은
`results/explore/rule_relax_20261101/측정계획.md`에 적어 두었습니다. 판정은 없습니다.

노트북은 3개(R·P1·P2)이고 **런타임을 하나씩 따로** 씁니다. 순서는 상관없습니다.

1. 런타임 유형 **CPU + 높은 RAM(CPU 8개)** 을 고르고 **런타임 → 모두 실행**을 누르세요. Drive 접근을 허용하세요.
2. 순서: 묶음 검증 → 패키지 → 시험 → 작은 실행(4판, 약 10분) → 본 실행(168판, CPU 8개 기준 약 3~4시간, 어림) → 비교표.
3. 본 실행은 백그라운드로 돕니다. 끊기면 다시 **모두 실행**하세요. 끝난 판은 건너뜁니다.
4. 세 노트북이 끝나면 "끝났어"라고 알려 주세요. 결과: Drive `FutureTraffic/explore_relax0/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
NAME = "P1"                                        # 이 노트북이 맡은 부분(R·P1·P2)
PAIRS = "P/A,P/B"
RUNS = 168
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_RELAX0.zip"
PLATOON0 = DRIVE / "explore_platoon0"                  # 현재 규칙의 M4·TM4(읽기만)
PBASE = DRIVE / "explore_platoon_base"                 # 고정신호·P0(읽기만)
ARENA3 = DRIVE / "arena3" / "referee" / "baseline"     # 무작위 세계 P0(읽기만)
ROOT_OUT = DRIVE / "explore_relax0"
OUT = ROOT_OUT / NAME
WORKSPACE = Path("/content/FutureTraffic_RELAX0")
MIN_CPUS = 8
for path in (ARCHIVE, PLATOON0 / "runs" / "P" / "TM4", PLATOON0 / "runs" / "R" / "M4", PBASE / "runs" / "P" / "FIXC", PBASE / "runs" / "R" / "FIXC",
             ARENA3 / "runs" / "P0"):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", OUT, "| 맡은 부분:", PAIRS)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "RELAX0_V1", "RELAX0 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("relax0_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON0 설정 {package_manifest['platoon0_config_sha256'][:12]} | 측정 계획 {package_manifest['plan_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

for name in ("test_explore_relax0.py", "test_explore_platoon0.py"):
    colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", name], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 작은 실행(약 10분): 첫 상황 4판(두 세계의 현재 규칙·A).
#    - 규칙을 안 바꾼 판이 기존 결과와 신호까지 같은가, A에서 좌회전 녹색이 실제로 짧아졌는가.
SMOKE = OUT / "smoke"
previous = colab.read_json(SMOKE / "smoke.json")
if previous and previous.get("ok"):
    print("이전 작은 실행이 통과해 건너뜁니다.")
else:
    colab.run_live(["src/explore_relax0.py", "smoke", "--out", str(SMOKE), "--work", "/content/rx_smoke",
                    "--workers", "4", "--platoon0", str(PLATOON0)], WORKSPACE, check=False)
s = colab.read_json(SMOKE / "smoke.json")
assert s and s.get("ok"), f"작은 실행 점검 실패 — 이 칸의 출력을 저에게 보내 주세요: {s and s.get('checks')}"
print("작은 실행 통과:", s["checks"])

In [ ]:
# 2) 본 실행. 백그라운드로 돌아 셀을 멈춰도 계속 돕니다. 런타임이 끊기면 처음 네 칸 → 이 칸을 다시 실행하세요. 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
LOGS = OUT / "logs"

def describe(name):
    done = len(list((OUT / "runs").glob("*/*/*/cache.json")))
    failed = len(list((OUT / "runs").glob("*/*/*.failed.json")))
    return f"끝난 판 {done}/{RUNS}, 실패 기록 {failed}"

pids = colab.launch({f"relax0_{NAME}": ["src/explore_relax0.py", "run", "--out", str(OUT), "--work", "/content/rx",
                                        "--workers", str(CPUS), "--pairs", PAIRS]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, "본 실행에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."

In [ ]:
# 3) 비교표(판정 없음). 끝난 규칙만 나옵니다. 숫자는 맥에서 다시 계산합니다.
colab.run_live(["src/explore_relax0.py", "report", "--out", str(ROOT_OUT), "--platoon0", str(PLATOON0),
                "--pbase", str(PBASE), "--arena3", str(ARENA3)], WORKSPACE, check=False)
if (ROOT_OUT / "summary.md").exists():
    display(Markdown((ROOT_OUT / "summary.md").read_text()))